# 🚕 TripPulse — Week 3: Source Profiling, Relationships and Architecture

**Project:** Team02 — TripPulse: Urban Mobility Analytics
**Databricks Volume path:** `/Volumes/trippulse/default/trippulsedata`
**Notebook path:** `notebooks/01_data_exploration.ipynb`

## Project story

TripPulse is a fictional ride-hailing platform used for this Data Engineering
internship. Riders request trips, drivers accept and complete them, and each
trip can trigger one or more payment attempts before a payment succeeds.
Zones describe the fictional city areas where trips start and end.

All data in this project is **synthetic educational data**. It does not
represent any real ride-hailing platform, driver, rider or location.

## Week-3 mission

This notebook profiles every TripPulse source file, proves primary-key and
foreign-key relationships, demonstrates the trip-to-payment one-to-many
overcount risk, and connects the source model to the approved TripPulse
architecture (P02-D02 source/entity map, P02-D03 ER diagram, and the C04
end-to-end lifecycle).

This notebook stops at:

```text
one TripPulse Bronze demo table
→ one TripPulse lineage demo view
```

The complete Bronze layer, ingestion framework, reconciliation, Silver
cleaning and Gold calculations all belong to **Week 4** and are not built
here.


## 🎯 Week-3 outcome

By the end of this notebook, every intern should be able to:

- find the uploaded TripPulse files in the Databricks Volume;
- read CSV, JSON (array) and Parquet sources with the correct Spark options;
- create PySpark DataFrames and Spark SQL temporary views for every source;
- inspect schema, grain and business keys for every source;
- explain the difference between physical rows and distinct business keys;
- profile categories, ranges, missing values and impossible values;
- prove primary-key and foreign-key relationships with left-anti joins;
- demonstrate the trip-to-payment one-to-many overcount risk;
- answer one business question at the correct grain;
- build exactly one Bronze demonstration table and one lineage demonstration view;
- explain the Week-3 versus Week-4 boundary.


## 🧩 Databricks cell languages

Databricks notebooks allow different cell languages inside the same notebook.

Use the cell-language dropdown, or a magic command at the top of a cell, to
choose:

- **Python** — used here for PySpark DataFrame reads, displays and simple
  counts.
- **SQL** (`%sql`) — used here as the **primary language** for profiling,
  relationship checks, aggregations and table/view creation.
- **File system** (`%fs`) — used to list files inside the Databricks Volume.

Most of this notebook is Spark SQL because SQL is the fastest, most readable
way to profile a source and prove a relationship. PySpark is used only where
the task requires it: reading files, creating DataFrames, and a small number
of simple counts.


## 🧭 Notebook map

| Section | What you will do |
|---|---|
| 1 | Confirm the TripPulse Volume and uploaded source files |
| 2A | Understand the trips.parquet nanosecond-timestamp issue |
| 2 | Create a PySpark DataFrame and Spark SQL view for every source |
| 3 | Inspect schemas |
| 4 | Display table contents |
| 5 | Understand grain and business keys |
| 6 | Count physical records |
| 7 | Compare physical rows with distinct business keys |
| 8 | Inspect category, date and numeric values |
| 9 | Find simple data concerns (missing, negative, impossible values) |
| 10 | Check timestamp anomalies |
| 11 | Check parent-child relationships |
| 12 | Demonstrate the trip-to-payment overcount risk |
| 13 | Answer one business question |
| 14 | Build one Bronze demonstration table |
| 15 | Inspect Delta detail and history |
| 16 | Build one lineage demonstration view |
| 17 | Close the Week-3 boundary and record evidence |


# 1. Prepare Databricks

Before running this notebook:

1. Open your Databricks workspace.
2. Attach **Serverless notebook compute**.
3. Confirm the TripPulse Volume exists at:

```text
/Volumes/trippulse/default/trippulsedata
```

4. Confirm these four Week-3 source files are uploaded to that Volume:

```text
zones.csv
drivers.json
trips.parquet
payments.csv
```

> This project also defines two streaming event-drop files
> (`ride_request_event_drop_01.json`, `ride_request_event_drop_02.json`) in
> the Week-2 data dictionary. They are **not** part of this notebook because
> they were not provided as part of this Week-3 Data Pack. Do not invent
> results for files that were not supplied.


## 💡 Tip — Know where things live

| Item | Correct place |
|---|---|
| Notebook | Databricks Workspace |
| Full data files | Unity Catalog Volume (`/Volumes/trippulse/default/trippulsedata`) |
| Screenshots | GitHub repository (`screenshots/`) |
| Weekly log | GitHub repository (`weekly_logs/week03_log.md`) |
| Full working dataset | Do not commit to GitHub |

A notebook contains instructions and code. The data itself stays in the
Volume, not in the repository.


# 2. Check the uploaded files

Before loading data, confirm that the four TripPulse source files are
visible in the Volume.


In [ ]:
%fs
ls /Volumes/trippulse/default/trippulsedata

### Expected files

```text
zones.csv
drivers.json
trips.parquet
payments.csv
```

If one is missing, stop and upload it before continuing.

> **Professional habit:** Always confirm the source files before writing
> queries. A missing file produces a confusing error much later in the
> notebook if you skip this step.


# 2A. A known TripPulse data issue: nanosecond timestamps in `trips.parquet`

`trips.parquet` stores its timestamp columns (`request_ts`,
`driver_accept_ts`, `pickup_ts`, `dropoff_ts`, `cancel_ts`,
`record_created_ts`) as Parquet `INT64 TIMESTAMP(NANOS)`. Spark 3.2 and
later, including current Databricks Runtime, cannot read that physical type
natively and fails with:

```text
[PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT64 (TIMESTAMP(NANOS,true))
```

Tools like pandas/pyarrow read nanosecond timestamps without issue, which
is why the file looks fine outside Databricks.

**Do not try `spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")`
on Serverless compute.** That legacy configuration key is not exposed on
Serverless and setting it fails with:

```text
[CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration
spark.sql.legacy.parquet.nanosAsLong is not available.
```

The fix used in this notebook (Section 3.3) is the Databricks-documented
workaround that does **not** depend on any session configuration: give
`spark.read` an explicit schema for `trips.parquet` in which the six
nanosecond timestamp columns are declared as `LongType`. Spark then skips
its own (failing) type inference for those columns and reads the raw
nanosecond values directly.


# 3. Create PySpark DataFrames and Spark SQL views


A Spark SQL view gives a file a simple, table-like name.

Instead of repeatedly referring to a long Volume path, we can write:

```sql
SELECT * FROM trips
```

We will create one PySpark DataFrame and one temporary SQL view for each of
the four TripPulse sources: `zones`, `drivers`, `trips`, `payments`.


## 3.1 Create the `zones` view

`zones.csv` is a plain CSV reference file. `header = true` tells Spark the
first row contains column names. `inferSchema = true` asks Spark to detect
common data types.


### What this Python block does

This cell:

1. reads the `zones.csv` reference file;
2. creates a PySpark DataFrame named `zones_df`;
3. creates a temporary SQL view named `zones`.

Use the cell-language dropdown and select **Python** before running it.


In [ ]:
# Load the zone reference file as a PySpark DataFrame
zones_df = spark.read.csv(
    "/Volumes/trippulse/default/trippulsedata/zones.csv",
    header=True,
    inferSchema=True
)

# Make the DataFrame available to Spark SQL
zones_df.createOrReplaceTempView("zones")

## 3.2 Create the `drivers` view

`drivers.json` is a pretty-printed **JSON array** (one array containing many
driver objects), not JSON Lines. Spark needs `multiLine = True` to read a
JSON array correctly. Using the default single-line JSON reader here would
either fail or silently load the wrong structure.


### What this Python block does

This cell reads the driver JSON array and creates both:

- a PySpark DataFrame named `drivers_df`;
- a Spark SQL temporary view named `drivers`.


In [ ]:
# Load the driver reference file (JSON array — requires multiLine)
drivers_df = spark.read.json(
    "/Volumes/trippulse/default/trippulsedata/drivers.json",
    multiLine=True
)

# Make it available to Spark SQL
drivers_df.createOrReplaceTempView("drivers")

## 3.3 Create the `trips` view

`trips.parquet` is the main transaction/event file: one row per ride
request. Parquet already carries an embedded schema, so no `header` or
`inferSchema` option is required — but an embedded schema still needs to be
checked against the approved data dictionary, not assumed to be correct.

As explained in **Section 2A**, this file's six timestamp columns are
stored as Parquet `INT64 TIMESTAMP(NANOS)`, which Spark cannot infer
automatically. The fix is to give `spark.read` an explicit schema in which
those six columns are declared as `LongType`. An explicit schema must cover
every column in the file — Spark cannot mix an explicit schema for some
columns with inference for the rest — so the full schema below is provided.
It matches `docs/data_dictionary.md` and the actual Parquet footer for this
file.


### What this Python block does

This cell:

1. defines an explicit schema for every `trips.parquet` column, using
   `LongType` for the six nanosecond timestamp columns;
2. reads the trip Parquet file into a PySpark DataFrame named `trips_df`
   using that schema;
3. creates a Spark SQL temporary view named `trips`.


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)

# Explicit schema — the six *_ts columns are nanosecond timestamps in the
# Parquet file, so they are declared as LongType instead of TimestampType.
# This avoids [PARQUET_TYPE_ILLEGAL] without needing any session config.
trips_schema = StructType([
    StructField("trip_id",               StringType(), True),
    StructField("request_ts",            LongType(),   True),
    StructField("driver_accept_ts",      LongType(),   True),
    StructField("pickup_ts",             LongType(),   True),
    StructField("dropoff_ts",            LongType(),   True),
    StructField("cancel_ts",             LongType(),   True),
    StructField("driver_id",             StringType(), True),
    StructField("pickup_zone_id",        StringType(), True),
    StructField("dropoff_zone_id",       StringType(), True),
    StructField("service_type",          StringType(), True),
    StructField("trip_status",           StringType(), True),
    StructField("cancellation_reason",   StringType(), True),
    StructField("estimated_distance_km", DoubleType(), True),
    StructField("actual_distance_km",    DoubleType(), True),
    StructField("estimated_fare_inr",    DoubleType(), True),
    StructField("final_fare_inr",        DoubleType(), True),
    StructField("surge_multiplier",      DoubleType(), True),
    StructField("record_created_ts",     LongType(),   True),
])

# Load the main trip request/lifecycle file with the explicit schema
trips_df = spark.read.schema(trips_schema).parquet(
    "/Volumes/trippulse/default/trippulsedata/trips.parquet"
)

# Make it available to Spark SQL
trips_df.createOrReplaceTempView("trips")

> **Read carefully after running this cell.** The six timestamp
> columns now load as **LongType** (raw nanoseconds since epoch), not
> `TimestampType`. Check `trips_df.printSchema()` in the next section to
> confirm this. Convert them explicitly before using them in date/time
> functions or in any of the timestamp-range/anomaly checks later in this
> notebook, for example:
>
> ```python
> from pyspark.sql.functions import col, from_unixtime
>
> trips_df = trips_df.withColumn(
>     "request_ts",
>     from_unixtime(col("request_ts") / 1_000_000_000).cast("timestamp")
> )
> ```
>
> Apply the same conversion to `driver_accept_ts`, `pickup_ts`, `dropoff_ts`,
> `cancel_ts` and `record_created_ts`, then re-run
> `createOrReplaceTempView("trips")` so the SQL view reflects the corrected
> types. Do this once, right after this cell, before moving on to schema
> inspection — every later section in this notebook assumes these columns
> behave as timestamps.


## 3.4 Create the `payments` view

`payments.csv` stores one row per **payment attempt**. A single trip can
have more than one payment attempt, so `payments` and `trips` do not share
the same grain.


### What this Python block does

This cell reads the payments CSV and creates both:

- a PySpark DataFrame named `payments_df`;
- a Spark SQL temporary view named `payments`.


In [ ]:
# Load the payment attempt file
payments_df = spark.read.csv(
    "/Volumes/trippulse/default/trippulsedata/payments.csv",
    header=True,
    inferSchema=True
)

# Make it available to Spark SQL
payments_df.createOrReplaceTempView("payments")

## 3.5 Confirm that the views exist

In [ ]:
%sql
SHOW TABLES;

You should see temporary views named:

```text
zones
drivers
trips
payments
```

These views exist for the current notebook session only.


# 3A. Confirm the created DataFrames

At this point, four PySpark DataFrames should exist:

```text
zones_df
drivers_df
trips_df
payments_df
```

Use the following short Python cell to display their names and column
counts.


In [ ]:
# Confirm the four DataFrames and their column counts
print("zones_df columns:", len(zones_df.columns))
print("drivers_df columns:", len(drivers_df.columns))
print("trips_df columns:", len(trips_df.columns))
print("payments_df columns:", len(payments_df.columns))

This is a simple existence check. It does not replace schema
inspection, row counts or table previews.


# 4. Inspect the schema

A schema describes the structure of a dataset. It tells us:

- column names;
- data types;
- possible identifiers;
- date and timestamp fields;
- numeric measures;
- nullable fields.

We will inspect each view separately and compare it with
`docs/data_dictionary.md`.


## 4.1 Zones schema

### PySpark method — print the DataFrame schema

This is the quickest way to inspect DataFrame columns and data types.


In [ ]:
# Show zone column names and data types
zones_df.printSchema()

### Spark SQL method — describe the SQL view

The SQL version displays the same structure in table form.


In [ ]:
%sql
DESCRIBE zones;

### What to notice

Look for:

- `zone_id` — the business key for the zone reference file;
- `zone_name`, `zone_type`, `city_code`, `demand_band` — descriptive/category fields;
- `is_active` — a boolean flag;
- `effective_from` — the date the zone definition became effective.


## 4.2 Drivers schema

In [ ]:
%sql
DESCRIBE drivers;

### What to notice

Look for:

- `driver_id` — likely business key;
- `home_zone_id` — relationship field to `zones`;
- `onboard_date`, `last_status_update_ts` — time fields;
- `driver_status`, `vehicle_type`, `service_type` — category fields;
- `rating`, `lifetime_completed_trips` — numeric fields;
- `source_record_version` — a versioning field, not a business measure.


## 4.3 Trips schema

In [ ]:
%sql
DESCRIBE trips;

### What to notice

Look for:

- `trip_id` — likely business key;
- `driver_id` — relationship field to `drivers`, expected to be conditional (null before assignment or for cancelled trips);
- `pickup_zone_id`, `dropoff_zone_id` — relationship fields to `zones`;
- `request_ts`, `driver_accept_ts`, `pickup_ts`, `dropoff_ts`, `cancel_ts`, `record_created_ts` — time/lifecycle fields;
- `trip_status`, `cancellation_reason`, `service_type` — category fields;
- `estimated_distance_km`, `actual_distance_km`, `estimated_fare_inr`, `final_fare_inr`, `surge_multiplier` — numeric fields.


## 4.4 Payments schema

In [ ]:
%sql
DESCRIBE payments;

### What to notice

Look for:

- `payment_id` — likely business key for one payment attempt;
- `trip_id` — relationship field to `trips` (one trip can have many payment attempts);
- `attempt_number` — the attempt sequence for a given trip;
- `payment_ts` — time field;
- `payment_method`, `payment_status`, `failure_reason` — category fields;
- `amount_inr` — numeric field;
- `is_final_attempt` — boolean flag;
- `payment_reference` — an external reference identifier, not necessarily unique per attempt.


## 💡 Tip — Compare with the Week-2 data dictionary

Open `docs/data_dictionary.md`.

Compare it manually with the actual schema output above. Ask:

- Are the expected columns present?
- Did Spark detect the expected data types?
- Is any column missing?
- Is any extra column present?
- Does the proposed business key actually exist and look unique?


# 5. Display the table contents

A schema tells us the structure. The actual rows tell us how the data
looks. Always inspect a few rows before writing analytical queries.


## 5.1 Display zone records

### PySpark method — display DataFrame rows

In [ ]:
# Display the first 10 zone records
display(zones_df.limit(10))

### Spark SQL method — display the same rows

The SQL query below reads from the temporary view created from the same
DataFrame.


In [ ]:
%sql
SELECT *
FROM zones
LIMIT 10;

## 5.2 Display driver records

### PySpark method — display the drivers DataFrame

In [ ]:
# Display the first 10 driver records
display(drivers_df.limit(10))

### Spark SQL method — display the drivers view

In [ ]:
%sql
SELECT *
FROM drivers
LIMIT 10;

## 5.3 Display trip records

### PySpark method — display the trips DataFrame

In [ ]:
# Display the first 10 trip records, most recent request first
display(trips_df.orderBy(trips_df.request_ts.desc()).limit(10))

### Spark SQL method — display the trips view

In [ ]:
%sql
SELECT *
FROM trips
ORDER BY request_ts DESC
LIMIT 10;

## 5.4 Display payment records

### PySpark method — display the payments DataFrame

In [ ]:
# Display the first 10 payment records
display(payments_df.limit(10))

### Spark SQL method — display the payments view

In [ ]:
%sql
SELECT *
FROM payments
LIMIT 10;

### Look carefully

Notice:

- how identifiers are formatted (`ZON-###`, `DRV-######`, `TRP-YYYYMMDD-######`, `PAY-#########`);
- how timestamps appear, and that several trip timestamps can legitimately be empty (`driver_accept_ts`, `pickup_ts`, `dropoff_ts`, `cancel_ts`);
- the values used in `trip_status`, `payment_status`, `payment_method`;
- whether `final_fare_inr` and `actual_distance_km` can be empty for trips that were not completed;
- whether a single `trip_id` reappears across multiple `payments` rows.


## 🧠 Intern checkpoint 1

Complete these statements:

```text
The main transaction/event file is ____________________.
One row in trips.parquet appears to represent ____________________.
The likely business key for trips is ____________________.
The main date field in trips is ____________________.
The main status field in trips is ____________________.
A single trip can have more than one row in ____________________.
```


# 6. Understand the grain

## What is grain?

**Grain means what one row represents.**

For TripPulse:

| View | Expected grain | Approved primary key | Approved relationship |
|---|---|---|---|
| `zones` | one fictional city zone | `zone_id` | referenced by `drivers.home_zone_id`, `trips.pickup_zone_id`, `trips.dropoff_zone_id` |
| `drivers` | one driver snapshot | `driver_id` | `home_zone_id` → `zones.zone_id` |
| `trips` | one ride request / trip lifecycle record | `trip_id` | `driver_id` → `drivers.driver_id` (conditional); `pickup_zone_id`, `dropoff_zone_id` → `zones.zone_id` |
| `payments` | one payment attempt for a trip | `payment_id` | `trip_id` → `trips.trip_id` (one trip : many attempts) |

`trips` is the central entity in this project. `payments` and any future
ride-event stream must never be counted as trip requests — they describe
attempts and lifecycle events **about** a trip, not additional trips.


# 7. Count the physical records

Start with a simple row count for each view.


## PySpark method — count one DataFrame

This counts the physical rows in the main `trips` DataFrame.


In [ ]:
# Count physical trip rows
trip_row_count = trips_df.count()
print("Physical trip rows:", trip_row_count)
# Run and record actual result.

## Spark SQL method — count all four views

The SQL query below produces a compact source summary.


In [ ]:
%sql
SELECT 'zones' AS source, COUNT(*) AS records FROM zones
UNION ALL
SELECT 'drivers', COUNT(*) FROM drivers
UNION ALL
SELECT 'trips', COUNT(*) FROM trips
UNION ALL
SELECT 'payments', COUNT(*) FROM payments;
-- Run and record actual result for each source.

This query tells us how many physical rows Spark loaded from each
source. Record the actual counts here after running the notebook — do not
assume the approximate row counts from `docs/synthetic_data_assumptions.md`
(`~150`, `~8,000`, `~250,000`, `~230,000`) are the exact figures for this
Data Pack.


# 8. Compare rows with distinct business keys

A physical row count is not always the same as a business record count.

`docs/synthetic_data_assumptions.md` states that a small percentage of
duplicate `trip_id` values may be intentionally introduced. Let us test
this directly.


## PySpark method — count distinct business keys

This uses simple DataFrame operations to count unique `trip_id` values.


In [ ]:
# Count distinct trip IDs
distinct_trip_count = (
    trips_df.select("trip_id")
            .distinct()
            .count()
)

print("Distinct trip IDs:", distinct_trip_count)
# Run and record actual result.

## Spark SQL method — compare both values together for every source

The SQL query below is more compact for analytical comparison.


In [ ]:
%sql
SELECT 'zones' AS source, COUNT(*) AS physical_rows, COUNT(DISTINCT zone_id) AS distinct_keys FROM zones
UNION ALL
SELECT 'drivers', COUNT(*), COUNT(DISTINCT driver_id) FROM drivers
UNION ALL
SELECT 'trips', COUNT(*), COUNT(DISTINCT trip_id) FROM trips
UNION ALL
SELECT 'payments', COUNT(*), COUNT(DISTINCT payment_id) FROM payments;
-- Run and record actual result for each source.

### How to interpret this table

- For `zones` and `drivers`, physical rows should equal distinct keys — each
  row is expected to describe one unique reference/master record.
- For `trips`, if physical rows are **higher** than distinct `trip_id`
  values, the file contains repeated trip identifiers. This is a controlled
  data-quality issue documented in the synthetic-data assumptions, expressed
  as `COUNT(*) − COUNT(DISTINCT trip_id)`.
- For `payments`, distinct `payment_id` should equal physical rows (each
  attempt has its own identifier), but distinct `trip_id` will be **lower**
  than physical rows, because one trip can have several payment attempts.
  That difference is expected and by design — it is not a defect.

> **Professional interpretation:** Record the actual physical-row and
> distinct-key figures after running this notebook, then state in one
> sentence what the difference means for each source.


### Payments: trip-level attempt count

This second check makes the one-to-many payment pattern explicit by
comparing physical payment rows with distinct trip references.


In [ ]:
%sql
SELECT
  COUNT(*)                AS physical_payment_rows,
  COUNT(DISTINCT trip_id) AS distinct_trips_with_payments
FROM payments;
-- Run and record actual result.

# 9. Display repeated business keys

Now identify a few repeated `trip_id` values in `trips`.


In [ ]:
%sql
SELECT
  trip_id,
  COUNT(*) AS occurrences
FROM trips
GROUP BY trip_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;
-- Run and record actual result.

## 🧠 Intern checkpoint 2

Explain this in one sentence:

```text
The physical row count for trips is higher than the distinct trip_id count
because ____________________________________________________________.
```

This is the first grain-related discovery for TripPulse.


# 10. Inspect important values

Before looking for errors, understand the normal values in the file.


## 10.1 Trip status distribution

### PySpark method — group and count

In [ ]:
# Count trips by trip_status
display(
    trips_df.groupBy("trip_status")
            .count()
            .orderBy("count", ascending=False)
)

### Spark SQL method — perform the same analysis

In [ ]:
%sql
SELECT
  trip_status,
  COUNT(*) AS records
FROM trips
GROUP BY trip_status
ORDER BY records DESC;
-- Run and record actual result.

### Why this matters

A distribution helps us understand:

- common categories;
- rare categories;
- possible spelling or casing differences;
- unexpected values not listed in `docs/synthetic_data_assumptions.md`
  (`Requested`, `Accepted`, `Picked Up`, `Completed`, `Cancelled`);
- whether one category dominates the data.


## 10.2 Trip service-type distribution

In [ ]:
%sql
SELECT
  service_type,
  COUNT(*) AS records
FROM trips
GROUP BY service_type
ORDER BY records DESC;
-- Run and record actual result.

## 10.3 Cancellation-reason distribution

In [ ]:
%sql
SELECT
  cancellation_reason,
  COUNT(*) AS records
FROM trips
WHERE cancellation_reason IS NOT NULL
GROUP BY cancellation_reason
ORDER BY records DESC;
-- Run and record actual result.

## 10.4 Zone type and demand-band distribution

In [ ]:
%sql
SELECT
  zone_type,
  demand_band,
  COUNT(*) AS zone_records
FROM zones
GROUP BY zone_type, demand_band
ORDER BY zone_records DESC;
-- Run and record actual result.

## 10.5 Driver status and vehicle-type distribution

In [ ]:
%sql
SELECT
  driver_status,
  vehicle_type,
  COUNT(*) AS driver_records
FROM drivers
GROUP BY driver_status, vehicle_type
ORDER BY driver_records DESC;
-- Run and record actual result.

## 10.6 Payment status and method distribution

In [ ]:
%sql
SELECT
  payment_method,
  payment_status,
  COUNT(*) AS payment_records
FROM payments
GROUP BY payment_method, payment_status
ORDER BY payment_records DESC;
-- Run and record actual result.

## 10.7 Trip request date range

In [ ]:
%sql
SELECT
  MIN(request_ts) AS first_request,
  MAX(request_ts) AS latest_request
FROM trips;
-- Run and record actual result.

### Why this matters

`docs/synthetic_data_assumptions.md` states the simulated time period is
**January 2026 – June 2026**. Compare the observed `request_ts` range with
this stated window.


## 10.8 Driver onboarding date range

In [ ]:
%sql
SELECT
  MIN(onboard_date) AS earliest_onboard_date,
  MAX(onboard_date) AS latest_onboard_date
FROM drivers;
-- Run and record actual result.

## 10.9 Payment timestamp range

In [ ]:
%sql
SELECT
  MIN(payment_ts) AS earliest_payment,
  MAX(payment_ts) AS latest_payment
FROM payments;
-- Run and record actual result.

## 10.10 Fare-value range

In [ ]:
%sql
SELECT
  MIN(final_fare_inr) AS minimum_fare,
  MAX(final_fare_inr) AS maximum_fare,
  AVG(final_fare_inr) AS average_fare
FROM trips;
-- Run and record actual result.

## 10.11 Distance-value range

In [ ]:
%sql
SELECT
  MIN(actual_distance_km) AS minimum_distance,
  MAX(actual_distance_km) AS maximum_distance,
  AVG(actual_distance_km) AS average_distance
FROM trips;
-- Run and record actual result.

## 10.12 Surge-multiplier range

`docs/synthetic_data_assumptions.md` states that surge-multiplier values
are expected to be greater than or equal to `1.0` in validated records.


In [ ]:
%sql
SELECT
  MIN(surge_multiplier) AS minimum_surge,
  MAX(surge_multiplier) AS maximum_surge,
  AVG(surge_multiplier) AS average_surge
FROM trips;
-- Run and record actual result.

## 10.13 Driver rating range

In [ ]:
%sql
SELECT
  MIN(rating) AS minimum_rating,
  MAX(rating) AS maximum_rating,
  AVG(rating) AS average_rating
FROM drivers;
-- Run and record actual result.

# 11. Find simple data concerns

Week 3 is about observation. We are not cleaning the data yet.

We are only asking:

> **What should the Week-4 and Week-5 pipeline handle?**

`docs/synthetic_data_assumptions.md` documents that cancelled and
unassigned trips may legitimately contain NULL values for driver
assignment, pickup time, drop-off time, travelled distance and final fare.
Treat those as expected nullability, not automatically as defects — confirm
with a status cross-check before calling something a data-quality issue.


## 11.1 Missing driver assignment

In [ ]:
%sql
SELECT COUNT(*) AS missing_driver_id
FROM trips
WHERE driver_id IS NULL;
-- Run and record actual result.

### Missing driver assignment, by trip status

This confirms whether missing `driver_id` values line up with cancelled or
unassigned trips, as expected in the assumptions document.


In [ ]:
%sql
SELECT
  trip_status,
  COUNT(*) AS trips_missing_driver
FROM trips
WHERE driver_id IS NULL
GROUP BY trip_status
ORDER BY trips_missing_driver DESC;
-- Run and record actual result.

## 11.2 Missing pickup or drop-off zone

In [ ]:
%sql
SELECT
  SUM(CASE WHEN pickup_zone_id  IS NULL THEN 1 ELSE 0 END) AS missing_pickup_zone,
  SUM(CASE WHEN dropoff_zone_id IS NULL THEN 1 ELSE 0 END) AS missing_dropoff_zone
FROM trips;
-- Run and record actual result.

## 11.3 Negative or impossible fare values

`docs/synthetic_data_assumptions.md` states negative or inconsistent fare
values may be introduced intentionally, checked with `final_fare_inr < 0`.


In [ ]:
%sql
SELECT COUNT(*) AS negative_final_fares
FROM trips
WHERE final_fare_inr < 0;
-- Run and record actual result.

## 11.4 Negative or impossible distance values

In [ ]:
%sql
SELECT COUNT(*) AS negative_actual_distance
FROM trips
WHERE actual_distance_km < 0;
-- Run and record actual result.

## 11.5 Surge multiplier below 1.0

The assumptions document states surge multiplier should be `>= 1.0` in
validated records. Values below `1.0` are worth flagging for review.


In [ ]:
%sql
SELECT COUNT(*) AS surge_below_one
FROM trips
WHERE surge_multiplier < 1.0;
-- Run and record actual result.

## 11.6 Negative payment amounts

In [ ]:
%sql
SELECT COUNT(*) AS negative_payment_amounts
FROM payments
WHERE amount_inr < 0;
-- Run and record actual result.

## 11.7 Failed payment attempts

`docs/synthetic_data_assumptions.md` states failed payment attempts are
intentionally simulated, checked with `payment_status = 'failed'`.


In [ ]:
%sql
SELECT COUNT(*) AS failed_payment_attempts
FROM payments
WHERE payment_status = 'failed';
-- Run and record actual result.

## 11.8 Display a few suspicious trip records

In [ ]:
%sql
SELECT
  trip_id,
  driver_id,
  trip_status,
  final_fare_inr,
  actual_distance_km,
  surge_multiplier
FROM trips
WHERE final_fare_inr < 0
   OR actual_distance_km < 0
   OR surge_multiplier < 1.0
LIMIT 20;
-- Run and record actual result.

## 🧠 Intern checkpoint 3

Choose one issue and explain:

```text
Issue:
Why it matters:
Which later week should handle it:
```

Suggested answer structure:

> "I found ________. It could affect ________. It should be handled during
> Silver or Data Quality work."


# 12. Check timestamp anomalies

`docs/synthetic_data_assumptions.md` documents that invalid ride-progression
timestamps may be intentionally introduced, checked with
`pickup_ts < request_ts OR dropoff_ts < pickup_ts`.


## 12.1 Pickup before request

In [ ]:
%sql
SELECT COUNT(*) AS pickup_before_request
FROM trips
WHERE pickup_ts IS NOT NULL
  AND pickup_ts < request_ts;
-- Run and record actual result.

## 12.2 Drop-off before pickup

In [ ]:
%sql
SELECT COUNT(*) AS dropoff_before_pickup
FROM trips
WHERE dropoff_ts IS NOT NULL
  AND pickup_ts   IS NOT NULL
  AND dropoff_ts < pickup_ts;
-- Run and record actual result.

## 12.3 Display a few suspicious timestamp sequences

In [ ]:
%sql
SELECT
  trip_id,
  request_ts,
  driver_accept_ts,
  pickup_ts,
  dropoff_ts,
  trip_status
FROM trips
WHERE (pickup_ts IS NOT NULL AND pickup_ts < request_ts)
   OR (dropoff_ts IS NOT NULL AND pickup_ts IS NOT NULL AND dropoff_ts < pickup_ts)
LIMIT 20;
-- Run and record actual result.

# 13. Check relationships between files

TripPulse defines these approved parent-child relationships (P02-D02 /
P02-D03):

```text
drivers.home_zone_id     → zones.zone_id
trips.driver_id          → drivers.driver_id   (conditional)
trips.pickup_zone_id     → zones.zone_id
trips.dropoff_zone_id    → zones.zone_id
payments.trip_id         → trips.trip_id
```

A good relationship means the referenced value exists in the related file.
We use **left** and **left-anti** joins so that orphan child rows remain
visible — never use an inner join for a relationship test, because an inner
join silently discards the failures we are trying to find.


## 13.1 Driver home-zone relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_driver_home_zone
FROM drivers d
LEFT JOIN zones z
  ON d.home_zone_id = z.zone_id
WHERE z.zone_id IS NULL;
-- Run and record actual result.

## 13.2 Trip driver relationship

`driver_id` is a conditional field on `trips` — it can legitimately be NULL
before assignment or for cancelled trips. This check tests only the
non-NULL `driver_id` values against `drivers`.


In [ ]:
%sql
SELECT COUNT(*) AS invalid_trip_driver_references
FROM trips t
LEFT JOIN drivers d
  ON t.driver_id = d.driver_id
WHERE t.driver_id IS NOT NULL
  AND d.driver_id IS NULL;
-- Run and record actual result.

## 13.3 Trip pickup-zone relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_pickup_zone_references
FROM trips t
LEFT JOIN zones z
  ON t.pickup_zone_id = z.zone_id
WHERE t.pickup_zone_id IS NOT NULL
  AND z.zone_id IS NULL;
-- Run and record actual result.

## 13.4 Trip drop-off-zone relationship

In [ ]:
%sql
SELECT COUNT(*) AS invalid_dropoff_zone_references
FROM trips t
LEFT JOIN zones z
  ON t.dropoff_zone_id = z.zone_id
WHERE t.dropoff_zone_id IS NOT NULL
  AND z.zone_id IS NULL;
-- Run and record actual result.

## 13.5 Payment-to-trip relationship

`docs/synthetic_data_assumptions.md` notes that invalid zone references are
tested with an anti-join against `zones.csv`; the same left-anti pattern
applies here to confirm every payment attempt belongs to a known trip.


In [ ]:
%sql
SELECT COUNT(*) AS invalid_payment_trip_references
FROM payments p
LEFT JOIN trips t
  ON p.trip_id = t.trip_id
WHERE t.trip_id IS NULL;
-- Run and record actual result.

## 13.6 Display a few invalid references (if any)

If any of the checks above returned a non-zero count, inspect a sample of
the offending rows before deciding how Week 4 should handle them.


In [ ]:
%sql
SELECT
  t.trip_id,
  t.pickup_zone_id,
  t.dropoff_zone_id,
  t.driver_id
FROM trips t
LEFT JOIN zones z1 ON t.pickup_zone_id  = z1.zone_id
LEFT JOIN zones z2 ON t.dropoff_zone_id = z2.zone_id
WHERE z1.zone_id IS NULL
   OR z2.zone_id IS NULL
LIMIT 20;
-- Run and record actual result.

> Zero invalid references is still valuable evidence, because it means
> the relationship was actually tested — not assumed.


# 14. Understand the join effect — the trip-to-payment overcount risk

`trips` and `payments` do not share the same grain: one trip can have many
payment attempts. Joining them without care will multiply trip rows.


## 14.1 Joined rows vs distinct trip requests

In [ ]:
%sql
SELECT
  COUNT(*)                  AS joined_rows,
  COUNT(DISTINCT t.trip_id) AS distinct_trip_requests
FROM trips t
LEFT JOIN payments p
  ON t.trip_id = p.trip_id;
-- Run and record actual result.

### Expected pattern

```text
joined_rows                >  distinct_trip_requests
```

If a trip has three payment attempts, that trip contributes three rows to
the joined result — but it is still exactly **one** trip request. Any KPI
that reports "number of trips" from this joined result, without using
`COUNT(DISTINCT trip_id)` or first de-duplicating on the trip grain, will
overstate real trip volume.

This demonstrates why relationship and grain checks must happen before
building dashboards.


## 14.2 Inner-join effect

An inner join keeps only matching records. Any trip with **zero** payment
attempts disappears entirely from an inner-joined result — which would be
the wrong choice for a "trips per day" KPI.


In [ ]:
%sql
SELECT COUNT(*) AS matched_trip_rows
FROM trips t
INNER JOIN payments p
  ON t.trip_id = p.trip_id;
-- Run and record actual result.

# 15. Ask one business question

The TripPulse operations lead asks:

> Which pickup zone generates the highest raw trip-request activity?

We join `trips` to `zones` on `pickup_zone_id` and count records at the
correct trip grain.


In [ ]:
%sql
SELECT
  z.zone_name,
  z.zone_type,
  COUNT(*)                  AS trip_records,
  COUNT(DISTINCT t.trip_id) AS distinct_trips
FROM trips t
LEFT JOIN zones z
  ON t.pickup_zone_id = z.zone_id
GROUP BY z.zone_name, z.zone_type
ORDER BY trip_records DESC;
-- Run and record actual result.

### How to read this result

Write:

```text
Highest-activity pickup zone:
Number of physical trip records:
Number of distinct trips:
One-line observation:
```

Then add this limitation:

> Known data issues (duplicate trip identifiers, invalid zone references,
> timestamp anomalies) have not yet been corrected, so this is an
> exploratory result — not a trusted Gold KPI.


# 16. Preview the Bronze idea

The Week-3 flow is:

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demonstration table
→ one lineage demonstration view
```

Only **one** Bronze demonstration table is created here, using the main
`trips` entity. The complete, multi-source Bronze layer belongs to Week 4.


## 16.1 Create one Bronze demo table

This managed Delta table uses the main `trips` entity, preserves the source
columns and adds only basic ingestion metadata. No deduplication,
correction or Silver transformation is performed.


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.trippulse_week03_bronze_demo_trips
USING DELTA
AS
SELECT
  *,
  current_timestamp() AS ingested_at,
  '/Volumes/trippulse/default/trippulsedata/trips.parquet' AS source_file
FROM trips;

> This is a Week-3 learning table — not the official TripPulse Bronze
> layer. Do not build the remaining `zones`, `drivers` or `payments`
> Bronze tables in this notebook.


# 17. Confirm and display the demo table

In [ ]:
%sql
SHOW TABLES IN workspace.default LIKE 'trippulse_week03_bronze_demo_trips';

In [ ]:
%sql
SELECT *
FROM workspace.default.trippulse_week03_bronze_demo_trips
LIMIT 10;

Look for:

```text
ingested_at
source_file
```


# 18. Perform one source-to-demo count check

Only one simple reconciliation check belongs to Week 3. Full reconciliation
is a Week-4 activity.


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM trips) AS source_rows,
  (SELECT COUNT(*) FROM workspace.default.trippulse_week03_bronze_demo_trips) AS demo_rows;
-- Run and record actual result.

Expected:

```text
source_rows = demo_rows
```

This is a simple Week-3 confidence check. Full source-to-Bronze
reconciliation for every source belongs to Week 4.


# 19. Inspect Delta table details

In [ ]:
%sql
DESCRIBE DETAIL workspace.default.trippulse_week03_bronze_demo_trips;

Notice fields such as `format`, `location`, `createdAt`, `lastModified`
and `numFiles`.


# 20. Inspect Delta table history

In [ ]:
%sql
DESCRIBE HISTORY workspace.default.trippulse_week03_bronze_demo_trips;

| Concept | Question answered |
|---|---|
| Schema | What columns and data types exist? |
| Relationship | How do business entities connect? |
| History | What operations changed this Delta table? |
| Lineage | Which governed objects feed or use another object? |


# 21. Create a lineage demonstration view

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.default.trippulse_week03_lineage_demo_view
AS
SELECT
  trip_id,
  driver_id,
  pickup_zone_id,
  dropoff_zone_id,
  service_type,
  trip_status,
  request_ts,
  final_fare_inr,
  ingested_at
FROM workspace.default.trippulse_week03_bronze_demo_trips;

In [ ]:
%sql
SELECT *
FROM workspace.default.trippulse_week03_lineage_demo_view
LIMIT 20;

The governed lineage path is:

```text
trippulse_week03_bronze_demo_trips
                ↓
trippulse_week03_lineage_demo_view
```


# 22. View lineage in Catalog Explorer

1. Click **Catalog**.
2. Open `workspace`.
3. Open `default`.
4. Select `trippulse_week03_lineage_demo_view`.
5. Open **Lineage**.
6. Choose **See lineage graph** when available.
7. Identify `trippulse_week03_bronze_demo_trips` as the upstream object.
8. Capture a screenshot of the lineage graph for
   `screenshots/week03_08_lineage.png`.


## 🧠 Intern checkpoint

Explain:

```text
Files → DataFrames → temporary views → exploration
→ one Bronze demo table → one lineage demo view
```

Also explain why the complete TripPulse Bronze layer is deferred to Week 4.


# 23. Week-3 boundary

## Completed in this notebook

- source-file inspection in the TripPulse Volume;
- PySpark DataFrame creation and display for `zones`, `drivers`, `trips`, `payments`;
- temporary SQL views for all four sources;
- schema, grain, physical-row and distinct-key checks;
- category, date and numeric-range profiling;
- missing-value, negative-value and timestamp-anomaly checks;
- primary-key and foreign-key relationship checks using left/left-anti joins;
- one trip-to-payment overcount demonstration;
- one business question, answered at the correct trip grain;
- exactly one Bronze demonstration table (`trippulse_week03_bronze_demo_trips`);
- one source-to-demo row-count check;
- `DESCRIBE DETAIL` and `DESCRIBE HISTORY` on the demo table;
- exactly one lineage demonstration view (`trippulse_week03_lineage_demo_view`);
- Catalog Explorer lineage verification steps.

## Explicitly NOT built here — reserved for Week 4

- official Bronze tables for `zones`, `drivers` and `payments`;
- a production ingestion framework;
- a complete reconciliation framework across all sources;
- schema rescue or schema-evolution handling;
- duplicate removal from `trips`;
- Silver cleaning and conformance;
- Gold aggregation/KPI calculations;
- Power BI outputs;
- streaming implementation for the ride-event drop files.

> Week 3 teaches how the TripPulse data behaves. Week 4 builds the
> repeatable Bronze ingestion pipeline on top of what was proven here.


# 24. Evidence checklist

```text
screenshots/week03_01_source_files.png
screenshots/week03_02_dataframes.png
screenshots/week03_03_schemas.png
screenshots/week03_04_grain_counts_values.png
screenshots/week03_05_relationship_checks.png
screenshots/week03_06_overcount_demo.png
screenshots/week03_07_bronze_demo.png
screenshots/week03_08_lineage.png
```

Also update:

```text
docs/data_dictionary.md
docs/synthetic_data_assumptions.md
docs/pipeline_walkthrough.md
weekly_logs/week03_log.md
```

A screenshot never replaces the working notebook, code, table, validation
query, commit history or weekly log. Keep screenshots only when they help a
reviewer see a schema, count, execution result or UI state.


# 25. Final intern defence

Every intern should be able to explain:

1. The four TripPulse sources, their DataFrames and their temporary views.
2. The grain and approved business key of each source.
3. Why physical rows and distinct business keys can differ for `trips` and
   for `payments`, and what each difference means.
4. The category, date and numeric values observed, and which values look
   unexpected.
5. Which relationships were tested, using left/left-anti joins, and why an
   inner join is the wrong tool for a relationship test.
6. How joining `trips` to `payments` can overstate trip volume, and why
   `COUNT(DISTINCT trip_id)` is the correct trip-level measure.
7. What a managed Delta table is, and what `ingested_at`/`source_file`
   add to the raw source.
8. Why exactly one Bronze demo table and one lineage demo view were built
   in Week 3 — and why the full Bronze layer, Silver cleaning, Gold
   calculations and Power BI outputs are Week-4-and-later work.


# 🎉 Week-3 TripPulse foundation complete

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demo table
→ one downstream lineage view
```

This is the correct Week-3 stopping point for TripPulse.


# 26. Conversion validation report

This report documents how this notebook was converted from the PageLoop
Week-3B template (`4_ZENAIZ_PageLoop_Week03B_FINAL_CUT_Project_Conversion.ipynb`)
into the TripPulse-specific Week-3 exploration notebook.

## PageLoop references remaining

```text
0 — no PageLoop-specific entity, filename, column, path, table, view,
    business question, result or explanation remains in this notebook.
    (loans / books / branches / loan_id / book_id / branch_id / pageloop /
    GrandCity Libraries were all replaced with TripPulse equivalents.)
```

## Assigned source files covered

```text
zones.csv       — covered (reference: zone)
drivers.json    — covered (reference: driver)
trips.parquet   — covered (main transaction/event entity: trip)
payments.csv    — covered (one-to-many child entity: payment attempt)
```

`ride_request_event_drop_01.json` and `ride_request_event_drop_02.json` are
defined in `docs/data_dictionary.md` (Week-2) but were **not** part of this
Week-3 Data Pack upload, so they are not profiled in this notebook. This
should be confirmed with the mentor before Week 4.

## DataFrames created

```text
zones_df
drivers_df
trips_df
payments_df
```

## SQL views created

```text
zones
drivers
trips
payments
```

## Relationship checks included

```text
drivers.home_zone_id  → zones.zone_id
trips.driver_id       → drivers.driver_id     (conditional, non-null only)
trips.pickup_zone_id  → zones.zone_id
trips.dropoff_zone_id → zones.zone_id
payments.trip_id      → trips.trip_id
```

## Bronze demo-table count

```text
1 — workspace.default.trippulse_week03_bronze_demo_trips
```

## Lineage-view count

```text
1 — workspace.default.trippulse_week03_lineage_demo_view
```

## Week-4 overlap check

```text
No official Bronze tables were built for zones, drivers or payments.
No ingestion framework, reconciliation framework, schema-rescue framework,
duplicate removal, Silver cleaning, Gold calculation, Power BI output or
streaming implementation was built in this notebook.
Result: PASS — no Week-4 scope was started early.
```

## Known-issue fix applied

```text
trips.parquet stores six timestamp columns as Parquet INT64
TIMESTAMP(NANOS), which Spark 3.2+/current Databricks Runtime cannot read
natively ([PARQUET_TYPE_ILLEGAL] Illegal Parquet type:
INT64 (TIMESTAMP(NANOS,true))).

First attempted fix — spark.conf.set("spark.sql.legacy.parquet.nanosAsLong",
"true") — is NOT usable on Serverless compute; that config key is not
exposed there and setting it fails with
[CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION].

Fix actually applied (Section 3.3): read trips.parquet with an explicit
StructType schema in which the six *_ts columns are declared as LongType.
This is the Databricks-documented workaround and requires no session
configuration, so it works on Serverless and all-purpose compute alike.
Under this schema the affected columns load as LongType (raw nanoseconds);
a follow-up cast to TimestampType is shown immediately after the read cell
in Section 3.3, and must be applied before any timestamp-range or
timestamp-anomaly check later in this notebook.
```

## Notebook structural validation result

```text
Project story and Week-3 mission           present
Databricks language explanation            present
Volume and source-file inspection          present
One DataFrame per Week-3 source            present (4 of 4)
Display of every DataFrame                 present (4 of 4)
Schema inspection                          present (4 of 4)
Temporary SQL views                        present (4 of 4)
Display of every SQL view                  present (4 of 4)
Grain and business-key explanation         present
Physical row counts                        present
Distinct business-key counts               present
Physical-vs-distinct explanation           present
Category/value distributions               present
Date ranges                                present
Numeric ranges                             present
Missing-value checks                       present
Negative/impossible-value checks           present
Timestamp anomaly checks                   present
Relationship checks                        present (5 checks)
Invalid-reference/anti-join checks         present
One meaningful business question           present
One Bronze demonstration table             present (exactly 1)
One source-to-demo row-count check         present (exactly 1)
DESCRIBE DETAIL                            present
DESCRIBE HISTORY                           present
One downstream lineage demonstration view  present (exactly 1)
Catalog Explorer lineage instructions      present
Evidence checklist                         present
Intern defence questions                   present
Clear Week-3 vs Week-4 boundary            present
Result: PASS
```

> All counts, distributions and check results in this notebook are marked
> **"Run and record actual result"** where Databricks execution is
> required. No row count, defect count or percentage has been fabricated —
> per the conversion rules, execution-dependent findings must be produced
> by running this notebook against the real TripPulse Volume, not assumed
> from this conversion.
